In [ ]:
# Load environment variables (ANTHROPIC_API_KEY) from .env
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from anthropic import Anthropic, Omit, omit
from anthropic.types import MessageParam

client = Anthropic()

In [ ]:
# Message helpers as before; chat() now also takes a temperature.

def add_user_message(messages: list[MessageParam], text: str) -> None:
    messages.append(MessageParam(role="user", content=text))

def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    messages.append(MessageParam(role="assistant", content=text))

def chat(
    messages: list[MessageParam],
    *,
    model: str = "claude-sonnet-4-5",
    max_tokens: int = 1000,
    system: str | Omit = omit,
    temperature: float = 1.0) -> str:
    """Send the message history and return the reply.

    model: which Claude model to use; trades off capability, speed, and cost.
    max_tokens: hard cap on generated tokens — a limit, not a target (truncates).
    system: top-level instruction setting the model's role, rules, and tone;
        passed separately from messages. Defaults to `omit` (field dropped).
    temperature: how random sampling is, from 0.0 to 1.0. Low (→0) is focused
        and near-deterministic (good for facts, code, extraction); high (→1) is
        more varied and creative. It changes *which* tokens get picked, not
        correctness.
    """
    response = client.messages.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        system=system,
        temperature=temperature
    )

    return next(block.text for block in response.content if block.type == "text")

In [ ]:
# Sample the same prompt a few times per temperature to see the pattern:
# temp=0.0 -> outputs barely change; temp=1.0 -> they vary noticeably.
messages: list[MessageParam] = []
add_user_message(messages, "Generate a one sentence movie idea")

for temperature in (0.0, 1.0):
    print(f"\n=== temperature={temperature} ===")
    for i in range(3):
        print(f"{i + 1}. {chat(messages, temperature=temperature)}")